# Lesson 11.2: Make task dependencies inspectable

One task collects a source and a later one drafts from it. The draft depends on that earlier result, so its instructions alone are insufficient. Trace the order in which the tasks run and what the program checks before it accepts each result into the next task.

[Companion lesson](../modules/11-agent-teams/02-code-workshop.md) · [Original-source walkthrough](../modules/11-agent-teams/05-source-walkthrough.md)

Run one cell at a time from top to bottom. The programme uses the fictional records shown in the code. No API key or model download is needed.

## Predict the result

Follow the order source identifier through the three tasks. Which contribution must finish immediately before the draft can start?

In [ ]:
def collect(context):
    return {"source_ids": ["order-17"], "reported_difference": 2}

def assess(context):
    evidence = context["collect"]
    return {
        "source_ids": evidence["source_ids"],
        "finding": "Customer and shipment accounts require reconciliation"
    }

def draft(context):
    finding = context["assess"]
    return {"source_ids": finding["source_ids"], "text": finding["finding"]}

TASKS = [
    {"id": "collect", "needs": [], "run": collect},
    {"id": "assess", "needs": ["collect"], "run": assess},
    {"id": "draft", "needs": ["assess"], "run": draft},
]

def run_tasks(tasks):
    results = {}
    for task in tasks:
        task_id = task["id"]
        if task_id in results:
            raise ValueError("Duplicate task ID")
        missing = [name for name in task["needs"] if name not in results]
        if missing:
            raise ValueError(f"Unfinished dependencies: {missing}")
        context = {name: results[name] for name in task["needs"]}
        output = task["run"](context)
        if not isinstance(output, dict) or not output.get("source_ids"):
            raise ValueError("Task result lacks source references")
        results[task_id] = output
    return results

results = run_tasks(TASKS)
print(results["draft"]["text"])
print(results["draft"]["source_ids"])

## Inspect the result

`print` displays values below the cell. When present, `assert` checks a specific expectation and raises an error if it fails.

In [ ]:
assert results["draft"]["source_ids"] == ["order-17"]

## Try a change

Reverse the first two tasks so the assessment is asked to run before collection. What should happen before any draft is produced?

In [ ]:
# Try the change here, or make a temporary edit in the first code cell.

<details>
<summary>Worked answer</summary>

The assessment requires the collection result. The workflow refuses the reversed order before running the assessment.

```python
try:
    run_tasks([TASKS[1], TASKS[0], TASKS[2]])
except ValueError as error:
    assert "collect" in str(error)
else:
    raise AssertionError("The assessment must wait for collection")
```

</details>